# Hippo Encoder Distillation\n\nGoogle Colab notebook for training the CLIP-style encoder distillation setup in this repo.

## Flow\n\n1. Mount Google Drive\n2. Clone or reuse the repo\n3. Install dependencies\n4. Point the config at your dataset\n5. Start training

In [ ]:
from google.colab import drive\n\ndrive.mount('/content/drive')

In [ ]:
from pathlib import Path\nimport os\n\nREPO_URL = 'https://github.com/YOUR_USERNAME/Hippo-encoder.git'\nWORKDIR = Path('/content/Hippo-encoder')\n\nif not WORKDIR.exists():\n    !git clone {REPO_URL} {WORKDIR}\n\n%cd /content/Hippo-encoder\n!git pull --ff-only || true

In [ ]:
%cd /content/Hippo-encoder\n!python -m pip install --upgrade pip\n!pip install -e .

In [ ]:
from pathlib import Path\nimport json\n\nDATA_ROOT = Path('/content/drive/MyDrive/hippo_encoder_data')\nIMAGE_ROOT = DATA_ROOT / 'images'\nTRAIN_JSONL = DATA_ROOT / 'train.jsonl'\nOUTPUT_DIR = Path('/content/drive/MyDrive/hippo_encoder_runs/distill-clip-tiny')\n\nIMAGE_ROOT.mkdir(parents=True, exist_ok=True)\nOUTPUT_DIR.mkdir(parents=True, exist_ok=True)\n\nprint('Dataset root:', DATA_ROOT)\nprint('Image root:', IMAGE_ROOT)\nprint('Train manifest:', TRAIN_JSONL)\nprint('Output dir:', OUTPUT_DIR)

In [ ]:
from pathlib import Path\n\nTRAIN_JSONL = Path('/content/drive/MyDrive/hippo_encoder_data/train.jsonl')\n\nif not TRAIN_JSONL.exists():\n    TRAIN_JSONL.parent.mkdir(parents=True, exist_ok=True)\n    TRAIN_JSONL.write_text(\n        '{"image":"example.jpg","text":"a red chair near the window"}\n',\n        encoding='utf-8',\n    )\n    print('Wrote sample manifest to', TRAIN_JSONL)\nelse:\n    print('Manifest already exists:', TRAIN_JSONL)

In [ ]:
from pathlib import Path\nimport json\n\nconfig = {\n    'teacher_model_name': 'openai/clip-vit-base-patch32',\n    'student_model_name': 'distilgpt2',\n    'dataset_jsonl': '/content/drive/MyDrive/hippo_encoder_data/train.jsonl',\n    'image_root': '/content/drive/MyDrive/hippo_encoder_data/images',\n    'output_dir': '/content/drive/MyDrive/hippo_encoder_runs/distill-clip-tiny',\n    'max_text_length': 64,\n    'image_size': 224,\n    'batch_size': 8,\n    'num_epochs': 1,\n    'learning_rate': 1e-4,\n    'weight_decay': 1e-2,\n    'log_every': 10,\n    'save_every': 500,\n    'num_workers': 2,\n    'teacher_image_weight': 1.0,\n    'teacher_text_weight': 1.0,\n    'hidden_state_weight': 0.2,\n    'contrastive_weight': 0.2,\n    'normalize_targets': True,\n    'gradient_clip_norm': 1.0,\n    'warmup_steps': 100,\n    'seed': 42\n}\n\nruntime_config_path = Path('/content/Hippo-encoder/configs/colab_run.json')\nruntime_config_path.parent.mkdir(parents=True, exist_ok=True)\nruntime_config_path.write_text(json.dumps(config, indent=2), encoding='utf-8')\nprint(runtime_config_path.read_text())

In [ ]:
%cd /content/Hippo-encoder\n!python -m hippo_encoder.train --config configs/colab_run.json

## Notes\n\n- Replace `YOUR_USERNAME` in `REPO_URL` with your actual GitHub account.\n- Put your real images in `/content/drive/MyDrive/hippo_encoder_data/images`.\n- Update the JSONL manifest so each row has `image` and `text`.\n- If you switch to a different student under 1B params, only the config needs to change.